# Round 8 (H13-H16) - Grounder Mechanism Diagnostics and A1/H13 Extraction Experiment

**Author**: Konrad Jelen (kj)<br>
**Date**: 2026-06-11

Three pre-registered mechanism hypotheses (see `experiments/grounding/HYPOTHESIS.md`, Round 8), each with a diagnostic kill-gate measured BEFORE building the mechanism:

- **A1** - SaT multilingual claim extraction (the shipped verb gate is English-only)
- **A2** - atomic-fact scoring through the frozen manifold (+ **H-B** alignment-profile features, shared gate)
- **H-C** - negation-scope mismatch feature for VitaminC

All logic lives in `experiments/grounding/mechanisms.py`; this notebook is the executable record. Private data (gold parquet, trace cache) stays outside the repo / in the gitignored forensics stash.

In [1]:
# imports - stdlib
import json
import sys
from pathlib import Path

# imports - third party
import pandas as pd

# imports - experiment module (single source of truth)
sys.path.insert(0, str(Path.cwd().parent / "experiments" / "grounding"))
import mechanisms as M

In [2]:
# configuration
P_HIGH_THRESHOLD = M.P_HIGH_THRESHOLD   # shipped high-tier decision threshold (0.40)
COMBINED = M.COMBINED                   # data/processed/grounding_combined.parquet (gitignored)
GOLD = M.GOLD                           # verified gold parquet (gitignored forensics stash)
print(f"threshold={P_HIGH_THRESHOLD}  combined={COMBINED.exists()}  gold={GOLD.exists()}")

threshold=0.4  combined=True  gold=True


## A1 gate - verb-gate rejection rate by language

Gold claims were produced BY the shipped `extract_claims()` (survivorship bias), so gold-recall is circular. The honest gate: on the 639 raw answer documents, what fraction of length-passing sentences does the English verb gate reject, per language?

In [3]:
a1 = M.diag_a1()
a1

traces: 639, missing from cache: 0


,n,accept_rate,reject_rate
lang,,,
en,7663,0.907608,0.092392
fr,584,0.881849,0.118151
es,261,0.720307,0.279693
nb,254,0.496063,0.503937
und,162,0.598765,0.401235
it,124,0.145161,0.854839
yo,119,1.000000,0.000000
nl,94,0.691489,0.308511
nn,69,0.594203,0.405797


## A2 / H-B gate - multi-sentence error concentration

Pre-registered kill: < 30% of private RAG errors multi-sentence OR multi/single error-rate ratio < 1.5.

In [4]:
a2 = M.diag_a2()
print(json.dumps(a2, indent=2, default=str))

{
  "corpus": "private_rag",
  "n": 2752,
  "n_errors": 381,
  "share_claims_multi": 0.2852470930232558,
  "share_errors_multi": 0.27034120734908135,
  "err_rate_multi": 0.13121019108280255,
  "err_rate_single": 0.14133197763091002,
  "sent_count_distribution": {
    "1": 1967,
    "2": 417,
    "3": 183,
    "4": 78,
    "5": 43,
    "6": 23,
    "7": 21,
    "8": 8,
    "9": 4,
    "10": 4,
    "11": 1,
    "12": 1,
    "13": 1,
    "14": 1
  },
  "err_rate_ratio": 0.9283828987765201,
  "gate_pass": false
}


## H-C gate - negation-cue asymmetry on VitaminC

Pre-registered kill: asymmetry < 25% of errors, or non-error asymmetry >= half the error rate.

In [5]:
hc = M.diag_c()
print(json.dumps(hc, indent=2))

{
  "n": 800,
  "n_errors": 244,
  "asym_rate_errors": 0.036885245901639344,
  "asym_rate_non_errors": 0.017985611510791366,
  "false_accepts": 119,
  "false_rejects": 125,
  "gate_pass": false
}


## A1 head-to-head - shipped vs gate-only vs SaT+gate

Three extractor variants over the same 639 answer documents. `gate_only` isolates the gate change (regex split kept); `sat_gate` adds SaT boundaries. Falsifier: inflation > 2x with no recall gain.

In [6]:
a1_eval = M.eval_a1()
print(json.dumps(a1_eval, indent=2))

{
  "n_docs": 639,
  "shipped": {
    "claims_per_doc": 13.03,
    "inflation_vs_shipped": 1.0,
    "gold_coverage": 1.0,
    "cov_by_lang": {
      "de-DE": 1.0,
      "en": 1.0,
      "es-ES": 1.0,
      "fr-FR": 1.0,
      "it-IT": 1.0,
      "nb-NO": 1.0,
      "nl-NL": 1.0,
      "pt-PT": 1.0,
      "sv-SE": 1.0
    },
    "top_langs": {
      "en": 6955,
      "fr": 515,
      "es": 188,
      "nb": 126,
      "yo": 119,
      "und": 97,
      "nl": 65,
      "nn": 41
    }
  },
  "gate_only": {
    "claims_per_doc": 14.74,
    "inflation_vs_shipped": 1.13,
    "gold_coverage": 0.9971,
    "cov_by_lang": {
      "de-DE": 1.0,
      "en": 0.996,
      "es-ES": 1.0,
      "fr-FR": 1.0,
      "it-IT": 1.0,
      "nb-NO": 1.0,
      "nl-NL": 1.0,
      "pt-PT": 1.0,
      "sv-SE": 1.0
    },
    "top_langs": {
      "en": 7559,
      "fr": 574,
      "es": 260,
      "nb": 252,
      "it": 122,
      "yo": 119,
      "nl": 94,
      "nn": 67
    }
  },
  "sat_gate": {
    "claims_per

## Conclusions

- **A1 KEPT** - verb gate rejects 9.2% of English vs nb 50.4% / it 85.5% / de 55.1%; the language-agnostic gate alone doubles Norwegian admissions and recovers Italian from zero at 1.13x inflation and 0.997 gold coverage; SaT boundaries add more (1.31x, 0.990 coverage)
- **A2 / H-B REJECTED pre-build** - errors do not concentrate in multi-sentence claims (ratio 0.93, needed > 1.5)
- **H-C REJECTED pre-build** - negation asymmetry in 3.7% of VitaminC errors (needed >= 25%)
- **Follow-ups** - sampled dual-judge precision pass on newly admitted sentences; gold v2 re-extraction (gold carries extractor survivorship bias)

Recorded in `RESULTS.md`, `BENCHMARK.md` and `HYPOTHESIS.md` (Round 8).